In [1]:
from dotenv import load_dotenv

load_dotenv()

True

## Semantic search

In [2]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("resources/acmecorp-employee-handbook.pdf")

data = loader.load()

print(data)

C:\Users\tjmja\AppData\Local\Temp\ipykernel_15476\4167041359.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
incorrect startxref pointer(1)
parsing for Object Streams


[Document(metadata={'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2025-11-20T23:23:16+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2025-11-20T23:23:16+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': 'resources/acmecorp-employee-handbook.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1'}, page_content='Employee Handbook\nNon-Disclosure Agreement (NDA) Policy\nEmployees must protect confidential information belonging to the company, its clients, and partners.\nThis includes, but is not limited to, product roadmaps, customer data, internal communications,\nproprietary algorithms, financial information, and unreleased features. Confidential information may not\nbe shared with unauthorized individuals inside or outside the organization. These obligations continue\nafter employment ends.\nWorkplace Conduct Policy\nEmployees must maintain a respectful, professional environment

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=200, add_start_index=True
)
all_splits = text_splitter.split_documents(data)

print(len(all_splits))

3


Embedding Models: https://docs.langchain.com/oss/python/integrations/text_embedding

In [7]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

In [8]:
from langchain_core.vectorstores import InMemoryVectorStore

vector_store = InMemoryVectorStore(embeddings)

In [9]:
ids = vector_store.add_documents(documents=all_splits)

In [10]:
results = vector_store.similarity_search(
    "How many days of vacation does an employee get in their first year?"
)

print(results[0])

page_content='prohibited. Violations may result in disciplinary action.
Paid Time Off (PTO) Policy
Full■time employees accrue PTO according to the following schedule:  0–1 years of service: 10 days
per year (0.833 days per month)  1–3 years of service: 15 days per year (1.25 days per month)  3+
years of service: 20 days per year (1.67 days per month) PTO may be used for vacation, personal
needs, or illness. Requests should be submitted in advance through the HR system unless related to
an emergency. Employees may carry over up to 5 unused PTO days per calendar year. Extended
absences exceeding 5 consecutive business days require manager approval.
Travel & Expense Policy
Employees may be reimbursed for reasonable and necessary expenses incurred during approved
business travel. This includes transportation, lodging, meals, and incidental expenses within
established limits. Receipts must be submitted within 14 days of travel. First-class travel, personal' metadata={'producer': 'ReportL

Setup (cells 7 and 8). You create the Gemini embedding model, which turns text into lists of numbers (vectors) that capture its meaning. You then create an empty in-memory vector store that will use that model to embed everything you give it.

Indexing (cell 9). `add_documents(all_splits)` takes the handbook chunks from the earlier text splitter step, converts each one into a vector with Gemini, and stores the vector next to the original text. The ids it returns are just identifiers for the stored chunks. This is the step that calls the Gemini API, so it's where rate-limit errors would appear.

Searching (cell 10). `similarity_search` embeds your question the same way, then returns the stored chunk whose vector is closest in meaning. The top result is the paid time off policy chunk, which lists 10 days for 0 to 1 years of service. That answers the vacation question even though the chunk says "PTO" and never uses the word "vacation", which shows it matched on meaning rather than keywords. The chunk also begins mid-sentence ("prohibited. Violations may...") because the splitter cuts at fixed sizes with overlap. The metadata at the end shows where it came from: page 0 of the handbook PDF, starting at character 812 (start_index).

## RAG Agent

In [11]:
from langchain.tools import tool

@tool
def search_handbook(query: str) -> str:
    """Search the employee handbook for information"""
    results = vector_store.similarity_search(query)
    return results[0].page_content

In [12]:
from langchain.agents import create_agent

agent = create_agent(
    model="google_genai:gemini-3.1-flash-lite",
    tools=[search_handbook],
    system_prompt="You are a helpful agent that can search the employee handbook for information."
    )

In [13]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="How many days of vacation does an employee get in their first year?")]}
)

In [14]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='How many days of vacation does an employee get in their first year?', additional_kwargs={}, response_metadata={}, id='5cf8d28f-e535-4276-903c-7cbd0833d226'),
              AIMessage(content=[], additional_kwargs={'function_call': {'name': 'search_handbook', 'arguments': '{"query": "vacation days first year"}'}, '__gemini_function_call_thought_signatures__': {'call_282933': 'EnMKcQFpFH0TyWHFPQDQ/8BdonMDlgxkZn1gb4KiMtk3rNWFpikN9t+96wvolSDgUxbvXEZIp+fDH0FcsZKRjg5PrPpgDfUdNCPgdXQTBtyUC2YzJiXtljjQ1FkTvXCnnJgKYgjwh9iaghI/jSbwkPqmy2ok'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0f688-a8ed-7dc2-8b1a-1b8488f31346-0', tool_calls=[{'name': 'search_handbook', 'args': {'query': 'vacation days first year'}, 'id': 'call_282933', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 73, 'output_tokens': 21, 'total_tokens': 94, 'i

In [15]:
print(response['messages'][-1].content[0]['text'])

Employees accrue 10 days of Paid Time Off (PTO) per year during their first year of service (at a rate of 0.833 days per month). This PTO can be used for vacation, personal needs, or illness.


In [ ]:
The basic idea is simple, and most of the real difficulty is in making it work well:
- Chunking: how you split the document strongly affects results, as we saw with the mid-sentence chunk.
- Retrieval quality: plain similarity search can miss things, so production systems often add keyword search, re-ranking, or metadata filters.
- Scale: InMemoryVectorStore disappears when the kernel stops, so real systems use a persistent vector database.
- Evaluation: you need to check whether it retrieved the right chunks and whether the answers are grounded in them, not made up.
- Messy sources: real documents come as scanned PDFs, tables and many file types, and parsing them takes work.